In [ ]:
# =============================================================================
# CELL 1: INSTALL DEPENDENCIES + BUILD BETTI-MATCHING-3D
# =============================================================================
import subprocess, sys, os, types
from pathlib import Path
import importlib.util as _ilu

# --- Paths to bundled wheels and metrics source ---
WHEELS_DIR = Path("/kaggle/input/datasets/sohier/vesuvius-metric-resources/wheels")
METRICS_DIR = Path("/kaggle/input/datasets/sohier/vesuvius-metric-resources/topological-metrics-kaggle")

# Fallback: check alternative locations
for alt in [
    Path("/kaggle/input/topological-metrics-kaggle"),
    Path("/kaggle/input/vesuvius-metrics"),
    Path("/kaggle/input/datasets/rajbalabala/topological-metrics-kaggle"),
]:
    if not WHEELS_DIR.exists() and (alt / "wheels").exists():
        WHEELS_DIR = alt / "wheels"
    if not METRICS_DIR.exists() and (alt / "topological-metrics-kaggle").exists():
        METRICS_DIR = alt / "topological-metrics-kaggle"
    if not METRICS_DIR.exists() and (alt / "src" / "topometrics").exists():
        METRICS_DIR = alt

print(f"Wheels dir: {WHEELS_DIR} (exists={WHEELS_DIR.exists()})")
print(f"Metrics dir: {METRICS_DIR} (exists={METRICS_DIR.exists()})")

# --- Install wheels offline (skip incompatible ones) ---
installed_from_wheel = set()
if WHEELS_DIR.exists():
    wheels = sorted(WHEELS_DIR.glob("*.whl"))
    print(f"\nInstalling {len(wheels)} wheels offline...")
    for w in wheels:
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--no-deps", "--quiet", str(w)],
            capture_output=True, text=True
        )
        if r.returncode == 0:
            installed_from_wheel.add(w.name.split("-")[0].replace("_", "-").lower())
        else:
            print(f"  SKIP (incompatible): {w.name}")
    print(f"Installed {len(installed_from_wheel)} wheels.")

# --- Fallback: install missing critical packages from PyPI ---
REQUIRED_PKGS = {
    "surface_distance": "surface-distance",
    "cc3d": "connected-components-3d",
    "skimage": "scikit-image",
    "tifffile": "tifffile",
    "imagecodecs": "imagecodecs",
}
pip_install = []
for mod_name, pip_name in REQUIRED_PKGS.items():
    try:
        __import__(mod_name)
    except ImportError:
        pip_install.append(pip_name)

if pip_install:
    print(f"\nInstalling from PyPI: {pip_install}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + pip_install,
        check=False
    )

# Ensure pybind11 + cmake for Betti build
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pybind11[global]", "cmake"],
    capture_output=True, check=False
)

# --- Build Betti-Matching-3D (C++ pybind11 module) ---
# Source is in read-only /kaggle/input, so build into /kaggle/working
BETTI_SRC = METRICS_DIR / "external" / "Betti-Matching-3D"
BUILD_DIR = Path("/kaggle/working/betti_build")

if BETTI_SRC.exists() and not list(BUILD_DIR.rglob("betti_matching*")):
    print(f"\nBuilding Betti-Matching-3D...")
    print(f"  Source: {BETTI_SRC}")
    print(f"  Build:  {BUILD_DIR}")
    BUILD_DIR.mkdir(parents=True, exist_ok=True)

    import sysconfig
    py_exe = sys.executable
    py_inc = sysconfig.get_config_var("INCLUDEPY") or sysconfig.get_paths().get("include", "")
    py_libdir = sysconfig.get_config_var("LIBDIR") or ""
    ldver = sysconfig.get_config_var("LDVERSION") or sysconfig.get_config_var("VERSION")

    py_lib = ""
    for ext in [".so", ".a"]:
        candidate = os.path.join(py_libdir, f"libpython{ldver}{ext}")
        if os.path.exists(candidate):
            py_lib = candidate
            break

    cmake_args = [
        "cmake", "-S", str(BETTI_SRC), "-B", str(BUILD_DIR),
        f"-DPython_EXECUTABLE={py_exe}",
        f"-DPython_INCLUDE_DIR={py_inc}",
        f"-DPYBIND11_FINDPYTHON=ON",
    ]
    if py_lib:
        cmake_args.append(f"-DPython_LIBRARY={py_lib}")

    print("  CMake configure...")
    r1 = subprocess.run(cmake_args, capture_output=True, text=True)
    if r1.returncode != 0:
        print(f"  CMake configure failed:\n{r1.stderr[-800:]}")
    else:
        print("  CMake build...")
        r2 = subprocess.run(
            ["cmake", "--build", str(BUILD_DIR), "--parallel"],
            capture_output=True, text=True
        )
        if r2.returncode != 0:
            print(f"  Build failed:\n{r2.stderr[-800:]}")
        else:
            print("  Betti-Matching-3D built successfully!")
elif not BETTI_SRC.exists():
    print(f"\nWARNING: Betti source not found at {BETTI_SRC}")
else:
    print(f"\nBetti-Matching-3D already built in {BUILD_DIR}")

# --- Add topometrics source to Python path ---
TOPO_SRC = METRICS_DIR / "src"
if TOPO_SRC.exists():
    if str(TOPO_SRC) not in sys.path:
        sys.path.insert(0, str(TOPO_SRC))
    print(f"\nAdded {TOPO_SRC} to sys.path")

# --- Pre-import betti_matching from our writable build dir ---
# The _bm_loader.py in topometrics looks for betti_matching at a hardcoded
# read-only path inside /kaggle/input/. We fix this by:
#   1. Loading betti_matching directly from our BUILD_DIR
#   2. Injecting a patched _bm_loader into sys.modules BEFORE topometrics imports
betti_mod = None
if BUILD_DIR.exists():
    # Method 1: Add build dir and all subdirs to sys.path, then import
    for dirpath in [BUILD_DIR] + [p.parent for p in BUILD_DIR.rglob("*.so")]:
        dp = str(dirpath)
        if dp not in sys.path:
            sys.path.insert(0, dp)
    try:
        import betti_matching as _bm_direct
        betti_mod = _bm_direct
        print(f"  betti_matching imported via sys.path")
    except ImportError:
        pass

    # Method 2: Explicit spec-based load from .so files
    if betti_mod is None:
        for so_file in sorted(BUILD_DIR.rglob("*.so")):
            if "betti" in so_file.name.lower():
                try:
                    spec = _ilu.spec_from_file_location("betti_matching", str(so_file))
                    if spec and spec.loader:
                        betti_mod = _ilu.module_from_spec(spec)
                        spec.loader.exec_module(betti_mod)
                        sys.modules["betti_matching"] = betti_mod
                        print(f"  betti_matching loaded from {so_file}")
                        break
                except Exception as e:
                    print(f"  Failed to load {so_file}: {e}")

    # List what's actually in BUILD_DIR for debugging
    if betti_mod is None:
        build_files = list(BUILD_DIR.rglob("*"))
        so_files = [f for f in build_files if f.suffix == '.so']
        print(f"  DEBUG: BUILD_DIR has {len(build_files)} files, {len(so_files)} .so files")
        for sf in so_files[:5]:
            print(f"    {sf}")

if betti_mod is not None:
    # Patch the _bm_loader so topometrics finds our pre-built module
    fake_loader = types.ModuleType("topometrics._bm_loader")
    fake_loader.load_betti_matching = lambda: betti_mod
    sys.modules["topometrics._bm_loader"] = fake_loader
    print("  _bm_loader patched to use writable build dir")

# --- Verify imports ---
HAVE_FULL_METRICS = False
try:
    from topometrics import compute_leaderboard_score
    print("\n*** topometrics imported successfully! ***")
    print("  compute_leaderboard_score available")
    HAVE_FULL_METRICS = True
except ImportError as e:
    print(f"\nWARNING: Could not import topometrics: {e}")
    print("  Will use SurfaceDice + VOI only (70% of LB)")

try:
    import cc3d
    ver = getattr(cc3d, '__version__', 'unknown')
    print(f"  cc3d {ver} available")
except Exception as e:
    print(f"  cc3d NOT available: {e}")

try:
    import surface_distance as sd
    print("  surface_distance available")
except Exception as e:
    print(f"  surface_distance NOT available: {e}")

print(f"\nPython version: {sys.version}")
print("Dependency setup complete.")

In [ ]:
# =============================================================================
# CELL 2: IMPORTS + CONFIG (8 model paths)
# =============================================================================
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'        # synchronous CUDA → accurate error traces
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import gc
import json
import time
import random
import warnings
import traceback
from pathlib import Path
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass, field
from itertools import product

import numpy as np
import pandas as pd
import tifffile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.ndimage import (
    binary_fill_holes, binary_closing, binary_opening,
    label as ndimage_label, generate_binary_structure
)
from skimage.morphology import remove_small_objects
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================
@dataclass
class CVConfig:
    # =========================================================================
    # DATA PATHS
    # =========================================================================
    DATA_ROOT: Path = Path("/kaggle/input/vesuvius-challenge-surface-detection")

    # =========================================================================
    # 8 MODEL CHECKPOINT PATHS
    # Fold 0: V11 architecture (features=[32,64,128,256,320,320], blocks=[1,2,3,4,6,6])
    # Fold 1: V11 architecture (same as fold 0)
    # =========================================================================
    FOLD0_BEST: Path = Path("/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/default/7/checkpoints_v11/fold_0/best_model.pth")
    FOLD0_CKPT_A: Path = Path("/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/default/7/checkpoints_v11/fold_0/checkpoint_epoch_460.pth")
    FOLD0_CKPT_B: Path = Path("/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/default/7/checkpoints_v11/fold_0/checkpoint_epoch_440.pth")
    FOLD0_CKPT_C: Path = Path("/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/default/7/checkpoints_v11/fold_0/checkpoint_epoch_420.pth")

    FOLD1_BEST: Path = Path("/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/v11_fold1/1/checkpoints_v11/fold_1/best_model.pth")
    FOLD1_CKPT_A: Path = Path("/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/v11_fold1/1/checkpoints_v11/fold_1/checkpoint_epoch_100.pth")
    FOLD1_CKPT_B: Path = Path("/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/v11_fold1/1/checkpoints_v11/fold_1/checkpoint_epoch_120.pth")
    FOLD1_CKPT_C: Path = Path("/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/v11_fold1/1/checkpoints_v11/fold_1/checkpoint_epoch_110.pth")

    # Architecture per fold
    FOLD0_FEATURES: List[int] = field(default_factory=lambda: [32, 64, 128, 256, 320, 320])
    FOLD0_N_BLOCKS: List[int] = field(default_factory=lambda: [1, 2, 3, 4, 6, 6])
    FOLD1_FEATURES: List[int] = field(default_factory=lambda: [32, 64, 128, 256, 320, 320])
    FOLD1_N_BLOCKS: List[int] = field(default_factory=lambda: [1, 2, 3, 4, 6, 6])

    # =========================================================================
    # FOLD SPLIT (must match training exactly)
    # =========================================================================
    N_FOLDS: int = 3
    CV_SEED: int = 42

    # =========================================================================
    # INFERENCE SETTINGS
    # =========================================================================
    PATCH_SIZE: Tuple[int, int, int] = (192, 192, 192)
    OVERLAP: float = 0.5
    USE_AMP: bool = True

    # =========================================================================
    # EVALUATION — controls runtime of Cells 8/9/10
    # 262 val volumes × full sweep = days of compute.
    # Set MAX_EVAL_SAMPLES=0 to evaluate ALL volumes (slow but exhaustive).
    # Default 40 gives statistically meaningful results in reasonable time.
    # =========================================================================
    MAX_EVAL_SAMPLES: int = 40

    # =========================================================================
    # OUTPUT
    # =========================================================================
    OUTPUT_DIR: Path = Path("/kaggle/working/cv_eval")
    PRED_DIR: Path = Path("/kaggle/working/cv_eval/predictions")

    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"

cfg = CVConfig()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cfg.PRED_DIR.mkdir(parents=True, exist_ok=True)

# Auto-detect data root
for candidate in [
    Path("/kaggle/input/datasets/rajbalabala/3d-volume-training-data"),
    Path("/kaggle/input/datasets/asharamkanderiwal/aimo-dataset"),
    Path("/kaggle/input/3d-volume-training-data"),
    Path("/kaggle/input/vesuvius-challenge-surface-detection"),
]:
    if (candidate / "train.csv").exists():
        cfg.DATA_ROOT = candidate
        break

N_GPUS = min(torch.cuda.device_count(), 2) if torch.cuda.is_available() else 0

print(f"Data root: {cfg.DATA_ROOT} (exists={(cfg.DATA_ROOT / 'train.csv').exists()})")
print(f"Output: {cfg.OUTPUT_DIR}")
print(f"Device: {cfg.DEVICE}")
print(f"GPUs: {N_GPUS}")
if torch.cuda.is_available():
    for i in range(N_GPUS):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} "
              f"({torch.cuda.get_device_properties(i).total_mem / 1e9:.1f} GB)")
print(f"MAX_EVAL_SAMPLES: {cfg.MAX_EVAL_SAMPLES} (0=all)")

# Group model paths by fold
FOLD_MODELS = {
    0: {
        'paths': [cfg.FOLD0_BEST, cfg.FOLD0_CKPT_A, cfg.FOLD0_CKPT_B, cfg.FOLD0_CKPT_C],
        'names': ['fold0_best', 'fold0_ckptA', 'fold0_ckptB', 'fold0_ckptC'],
        'features': cfg.FOLD0_FEATURES,
        'n_blocks': cfg.FOLD0_N_BLOCKS,
    },
    1: {
        'paths': [cfg.FOLD1_BEST, cfg.FOLD1_CKPT_A, cfg.FOLD1_CKPT_B, cfg.FOLD1_CKPT_C],
        'names': ['fold1_best', 'fold1_ckptA', 'fold1_ckptB', 'fold1_ckptC'],
        'features': cfg.FOLD1_FEATURES,
        'n_blocks': cfg.FOLD1_N_BLOCKS,
    },
}

print(f"\nFold 0: features={cfg.FOLD0_FEATURES}, blocks={cfg.FOLD0_N_BLOCKS}")
print(f"Fold 1: features={cfg.FOLD1_FEATURES}, blocks={cfg.FOLD1_N_BLOCKS}")
print(f"Inference: patch={cfg.PATCH_SIZE}, overlap={cfg.OVERLAP}")

In [ ]:
# =============================================================================
# CELL 3: REPRODUCE EXACT FOLD SPLITS
# =============================================================================

def make_stratified_volume_folds(csv_path, images_dir, labels_dir, n_splits=3, seed=42):
    """Reproduce EXACT same folds as training (StratifiedKFold on scroll_id)."""
    df = pd.read_csv(csv_path)
    valid_mask = df['id'].apply(
        lambda x: (images_dir / f"{x}.tif").exists() and (labels_dir / f"{x}.tif").exists()
    )
    df = df[valid_mask].reset_index(drop=True)
    print(f"Found {len(df)} valid volumes")

    if 'scroll_id' in df.columns:
        strat_col = df['scroll_id'].values
    else:
        strat_col = df['id'].apply(lambda x: x.split('_')[0] if '_' in x else 'unknown').values
    print(f"Scroll distribution: {pd.Series(strat_col).value_counts().to_dict()}")

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(df['id'], strat_col)):
        train_ids = df.iloc[train_idx]['id'].tolist()
        val_ids = df.iloc[val_idx]['id'].tolist()
        assert len(set(train_ids) & set(val_ids)) == 0
        splits.append((train_ids, val_ids))
        print(f"  Fold {fold}: Train={len(train_ids)}, Val={len(val_ids)} → {val_ids}")
    return splits

train_csv = cfg.DATA_ROOT / "train.csv"
train_images = cfg.DATA_ROOT / "train_images"
train_labels = cfg.DATA_ROOT / "train_labels"

FOLD_SPLITS = []
if train_csv.exists():
    FOLD_SPLITS = make_stratified_volume_folds(
        train_csv, train_images, train_labels,
        n_splits=cfg.N_FOLDS, seed=cfg.CV_SEED
    )
    print(f"\nFold 0 val volumes: {FOLD_SPLITS[0][1]}")
    print(f"Fold 1 val volumes: {FOLD_SPLITS[1][1]}")
    if len(FOLD_SPLITS) > 2:
        print(f"Fold 2 val volumes: {FOLD_SPLITS[2][1]} (no models for this fold)")
else:
    print("ERROR: train.csv not found! Check DATA_ROOT path.")

In [ ]:
# =============================================================================
# CELL 4: MODEL ARCHITECTURE (TopoPreservingUNet3D — same as training)
# =============================================================================

def get_num_groups(channels, max_groups=32):
    for g in [max_groups, 16, 8, 4, 2, 1]:
        if channels % g == 0:
            return g
    return 1

class HybridConv3d(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        mid_ch = out_ch // 2
        self.conv_xy = nn.Conv3d(in_ch, mid_ch, kernel_size=(1,3,3), padding=(0,1,1), bias=False)
        self.conv_z = nn.Conv3d(in_ch, out_ch - mid_ch, kernel_size=(3,1,1), padding=(1,0,0), bias=False)
        self.norm = nn.GroupNorm(get_num_groups(out_ch), out_ch)
        self.act = nn.LeakyReLU(0.01, inplace=True)
    def forward(self, x):
        return self.act(self.norm(torch.cat([self.conv_xy(x), self.conv_z(x)], dim=1)))

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_hybrid=False):
        super().__init__()
        if use_hybrid:
            self.conv = HybridConv3d(in_ch, out_ch)
        else:
            self.conv = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.GroupNorm(get_num_groups(out_ch), out_ch),
                nn.LeakyReLU(0.01, inplace=True),
            )
    def forward(self, x): return self.conv(x)

class MultiScaleResBlock(nn.Module):
    def __init__(self, channels, scales=2, use_hybrid=False):
        super().__init__()
        self.scales = scales
        self.width = channels // scales
        self.convs = nn.ModuleList([ConvBlock(self.width, self.width, use_hybrid) for _ in range(scales-1)])
        self.norm = nn.GroupNorm(get_num_groups(channels), channels)
    def forward(self, x):
        splits = torch.chunk(x, self.scales, dim=1)
        outputs = [splits[0]]
        for i, conv in enumerate(self.convs):
            out = conv(splits[i+1] + outputs[-1]) if i > 0 else conv(splits[i+1])
            outputs.append(out)
        return x + self.norm(torch.cat(outputs, dim=1))

class AttentionBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool3d(1)
        self.fc1 = nn.Linear(channels, max(channels // reduction, 8))
        self.fc2 = nn.Linear(max(channels // reduction, 8), channels)
        self.conv_spatial = nn.Conv3d(2, 1, kernel_size=7, padding=3, bias=False)
    def forward(self, x):
        b, c = x.shape[:2]
        ca = torch.sigmoid(self.fc2(F.relu(self.fc1(self.gap(x).view(b, c))))).view(b, c, 1, 1, 1)
        x_ca = x * ca
        sa = torch.sigmoid(self.conv_spatial(torch.cat([x_ca.mean(1, keepdim=True), x_ca.max(1, keepdim=True)[0]], 1)))
        return x_ca * sa

class SurfaceRefinementBlock(nn.Module):
    """With depth tiling for OOM safety on T4."""
    TILE_DEPTH = 32
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.edge_conv = nn.Conv3d(in_ch, in_ch, 3, padding=1, bias=False)
        self.refine = nn.Sequential(
            nn.Conv3d(in_ch * 2, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(get_num_groups(out_ch), out_ch),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(get_num_groups(out_ch), out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )
    def _forward_tiled(self, x):
        B, C, D, H, W = x.shape
        assert B == 1
        tile = self.TILE_DEPTH
        pad = 1
        output_slices = []
        for d_start in range(0, D, tile):
            d_end = min(d_start + tile, D)
            d_start_pad = max(0, d_start - pad)
            d_end_pad = min(D, d_end + pad)
            x_tile = x[:, :, d_start_pad:d_end_pad, :, :]
            edge_tile = torch.abs(self.edge_conv(x_tile))
            cat_tile = torch.cat([x_tile, edge_tile], dim=1)
            del x_tile, edge_tile
            out_tile = self.refine(cat_tile)
            del cat_tile
            valid_start = d_start - d_start_pad
            valid_end = valid_start + (d_end - d_start)
            output_slices.append(out_tile[:, :, valid_start:valid_end, :, :])
            del out_tile
        return torch.cat(output_slices, dim=2)
    def forward(self, x):
        D = x.shape[2]
        if D <= self.TILE_DEPTH:
            return self.refine(torch.cat([x, torch.abs(self.edge_conv(x))], dim=1))
        outputs = []
        for i in range(x.shape[0]):
            outputs.append(self._forward_tiled(x[i:i+1]))
        return torch.cat(outputs, dim=0)

class TopoPreservingUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, features=None, n_blocks=None,
                 use_attention=True, use_hybrid_conv=True, use_surface_refinement=True):
        super().__init__()
        features = features or [32, 64, 128, 256, 320, 320]
        n_blocks = n_blocks or [1, 2, 3, 4, 6, 6]
        self.features = features
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i-1]
            layers = [ConvBlock(in_channels, feat, use_hybrid=use_hybrid_conv and i > 0)]
            layers.extend([MultiScaleResBlock(feat, 2, use_hybrid_conv) for _ in range(nb)])
            self.encoders.append(nn.Sequential(*layers))
            self.attentions.append(AttentionBlock(feat) if use_attention and i >= 2 else nn.Identity())
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, 2, stride=2))
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        for i in range(len(features)-2, -1, -1):
            self.ups.append(nn.ConvTranspose3d(features[i+1], features[i], 2, stride=2))
            if use_surface_refinement and i == 0:
                self.dec_convs.append(SurfaceRefinementBlock(features[i]*2, features[i]))
            else:
                self.dec_convs.append(nn.Sequential(
                    ConvBlock(features[i]*2, features[i], use_hybrid_conv),
                    MultiScaleResBlock(features[i], 2, use_hybrid_conv),
                ))
        self.final = nn.Conv3d(features[0], out_ch, 1)
    def forward(self, x):
        enc_features = []
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = att(enc(x))
            enc_features.append(x)
            if i < len(self.pools): x = self.pools[i](x)
        enc_features = enc_features[::-1]
        x = enc_features[0]
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i+1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = dec(torch.cat([x, skip], dim=1))
        return self.final(x)

# Verify both architectures
for name, feat, blk in [("V11 (fold 0)", cfg.FOLD0_FEATURES, cfg.FOLD0_N_BLOCKS),
                          ("V11 (fold 1)", cfg.FOLD1_FEATURES, cfg.FOLD1_N_BLOCKS)]:
    m = TopoPreservingUNet3D(features=feat, n_blocks=blk)
    n_params = sum(p.numel() for p in m.parameters()) / 1e6
    print(f"{name}: {n_params:.1f}M params | features={feat} | blocks={blk}")
    del m
print("Model architecture defined (with OOM-safe tiled SurfaceRefinementBlock)")

In [ ]:
# =============================================================================
# CELL 5: NORMALIZATION + SLIDING WINDOW INFERENCE
# =============================================================================

def robust_zscore_normalize(img, lower_percentile=0.5, upper_percentile=99.5):
    """Same as training — percentile clipping + Z-score."""
    p_low = np.percentile(img, lower_percentile)
    p_high = np.percentile(img, upper_percentile)
    img_clipped = np.clip(img, p_low, p_high)
    mean = img_clipped.mean()
    std = img_clipped.std()
    return ((img_clipped - mean) / (std + 1e-8)).astype(np.float32)


def create_gaussian_weight(patch_size, sigma=0.125):
    d, h, w = patch_size
    gz = np.exp(-0.5 * ((np.arange(d) - d/2) / (d * sigma)) ** 2)
    gy = np.exp(-0.5 * ((np.arange(h) - h/2) / (h * sigma)) ** 2)
    gx = np.exp(-0.5 * ((np.arange(w) - w/2) / (w * sigma)) ** 2)
    return (gz[:, None, None] * gy[None, :, None] * gx[None, None, :]).astype(np.float32)


def get_patch_positions(volume_shape, patch_size, overlap=0.5):
    D, H, W = volume_shape
    pd, ph, pw = patch_size
    sd = max(1, int(pd * (1 - overlap)))
    sh = max(1, int(ph * (1 - overlap)))
    sw = max(1, int(pw * (1 - overlap)))
    def _pos(total, patch, stride):
        if total <= patch: return [0]
        pos = list(range(0, total - patch + 1, stride))
        if (total - patch) not in pos: pos.append(total - patch)
        return pos
    positions = []
    for z in _pos(D, pd, sd):
        for y in _pos(H, ph, sh):
            for x in _pos(W, pw, sw):
                positions.append((z, y, x))
    return positions


@torch.no_grad()
def sliding_window_inference(model, volume_normalized, patch_size, overlap=0.5,
                              device="cuda", verbose=True):
    """Sliding window inference → probability map (float32 numpy).
    
    Args:
        verbose: If False, suppress tqdm (for multi-GPU threading).
    """
    model.eval()
    D, H, W = volume_normalized.shape
    pd, ph, pw = patch_size
    # Pad if needed
    pad_d = max(0, pd - D)
    pad_h = max(0, ph - H)
    pad_w = max(0, pw - W)
    if pad_d > 0 or pad_h > 0 or pad_w > 0:
        volume_normalized = np.pad(volume_normalized, ((0,pad_d),(0,pad_h),(0,pad_w)), mode='reflect')
        D, H, W = volume_normalized.shape

    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    weight_sum = np.zeros((D, H, W), dtype=np.float32)
    gauss = create_gaussian_weight(patch_size)
    positions = get_patch_positions((D, H, W), patch_size, overlap)
    if verbose:
        print(f"    {len(positions)} patches, overlap={overlap}")

    iterator = tqdm(positions, desc="  Infer", leave=False) if verbose else positions
    for idx, (z, y, x) in enumerate(iterator):
        patch = volume_normalized[z:z+pd, y:y+ph, x:x+pw]
        actual = patch.shape
        if actual != (pd, ph, pw):
            p = [(0, max(0, pd-actual[0])), (0, max(0, ph-actual[1])), (0, max(0, pw-actual[2]))]
            patch = np.pad(patch, p, mode='reflect')

        batch = torch.from_numpy(patch[None, None]).to(device).half()
        with torch.amp.autocast('cuda', dtype=torch.float16):
            pred = torch.sigmoid(model(batch))
        pred_np = pred[0, 0].float().cpu().numpy()

        pred_sum[z:z+pd, y:y+ph, x:x+pw] += pred_np * gauss
        weight_sum[z:z+pd, y:y+ph, x:x+pw] += gauss
        del batch, pred, pred_np
        if idx % 20 == 0: torch.cuda.empty_cache()

    result = pred_sum / np.maximum(weight_sum, 1e-8)
    # Remove padding
    orig_D = D - pad_d if pad_d > 0 else D
    orig_H = H - pad_h if pad_h > 0 else H
    orig_W = W - pad_w if pad_w > 0 else W
    return result[:orig_D, :orig_H, :orig_W]


def load_model_from_checkpoint(checkpoint_path, features, n_blocks, device='cpu'):
    """Load a single model from checkpoint to CPU."""
    model = TopoPreservingUNet3D(features=features, n_blocks=n_blocks)
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    state = ckpt.get('model_state_dict', ckpt)
    state = {k.replace('_orig_mod.', '').replace('module.', ''): v for k, v in state.items()}
    model.load_state_dict(state, strict=False)
    epoch = ckpt.get('epoch', '?')
    dice = ckpt.get('best_dice', 0)
    del ckpt, state
    gc.collect()
    model.eval()
    return model, epoch, dice

print("Inference functions ready")
print("  robust_zscore_normalize: same as training")
print("  sliding_window_inference: Gaussian blending, batch=1, float16")
print("  verbose=False available for multi-GPU threading")

In [ ]:
# =============================================================================
# CELL 6: COMPETITION METRIC + POST-PROCESSING FUNCTIONS
# (Moved BEFORE inference so inline scoring works during OOF generation)
# =============================================================================

def compute_competition_score(pred_binary, gt_labels, ignore_label=2):
    """
    Compute EXACT competition leaderboard score.
    LB = 0.30 × TopoScore + 0.35 × SurfaceDice + 0.35 × VOI_score
    """
    if HAVE_FULL_METRICS:
        report = compute_leaderboard_score(
            predictions=pred_binary,
            labels=gt_labels,
            dims=(0, 1, 2),
            spacing=(1.0, 1.0, 1.0),
            surface_tolerance=2.0,
            voi_connectivity=26,
            combine_weights=(0.3, 0.35, 0.35),
            fg_threshold=None,
            ignore_label=ignore_label,
        )
        return {
            'lb_score': report.score,
            'toposcore': report.topo.toposcore,
            'topoF1_0': report.topo.topoF1_by_dim.get(0, float('nan')),
            'topoF1_1': report.topo.topoF1_by_dim.get(1, float('nan')),
            'topoF1_2': report.topo.topoF1_by_dim.get(2, float('nan')),
            'surface_dice': report.surface_dice,
            'voi_score': report.voi.voi_score,
            'voi_total': report.voi.voi_total,
            'voi_split': report.voi.voi_split,
            'voi_merge': report.voi.voi_merge,
        }
    else:
        # Fallback: SurfaceDice + VOI only (70% of LB)
        import surface_distance as sd
        import cc3d as _cc3d

        ign = (gt_labels == ignore_label)
        pr = pred_binary.copy()
        gt = (gt_labels == 1).astype(np.uint8)
        pr[ign] = 0
        gt[ign] = 0

        # Surface Dice
        if not gt.any() and not pr.any():
            surf_dice = 1.0
        elif gt.any() ^ pr.any():
            surf_dice = 0.0
        else:
            sdists = sd.compute_surface_distances(gt.astype(bool), pr.astype(bool), (1,1,1))
            surf_dice = float(sd.compute_surface_dice_at_tolerance(sdists, 2.0))

        # VOI
        vs, vm = 0.0, 0.0
        gt_lab = _cc3d.connected_components(gt, connectivity=26)
        pr_lab = _cc3d.connected_components(pr, connectivity=26)
        union = (gt_lab > 0) | (pr_lab > 0)
        if union.any():
            from skimage.metrics import variation_of_information
            vs, vm = variation_of_information(gt_lab[union], pr_lab[union])
            voi_total = vs + vm
            voi_score = 1.0 / (1.0 + voi_total)
        else:
            voi_total = 0.0
            voi_score = 1.0

        topo_approx = 0.5
        lb_approx = 0.30 * topo_approx + 0.35 * surf_dice + 0.35 * voi_score

        return {
            'lb_score': lb_approx,
            'toposcore': topo_approx,
            'topoF1_0': float('nan'),
            'topoF1_1': float('nan'),
            'topoF1_2': float('nan'),
            'surface_dice': surf_dice,
            'voi_score': voi_score,
            'voi_total': voi_total,
            'voi_split': vs,
            'voi_merge': vm,
        }


# =============================================================================
# POST-PROCESSING (parameterized for sweep)
# =============================================================================

def count_components(mask):
    struct = generate_binary_structure(3, 3)
    _, n = ndimage_label(mask, structure=struct)
    return n

def remove_small_components(mask, min_size=50):
    struct = generate_binary_structure(3, 3)
    labeled, n = ndimage_label(mask, structure=struct)
    if n == 0: return mask
    sizes = np.bincount(labeled.ravel())
    small = sizes < min_size
    small[0] = False
    result = mask.copy()
    result[small[labeled]] = 0
    return result

def topology_safe_op(mask, op_func, name="op"):
    n_before = count_components(mask)
    result = op_func(mask)
    n_after = count_components(result)
    if n_after < n_before:
        return mask
    return result

def slicewise_hole_fill(mask):
    filled = mask.copy()
    for ax in range(3):
        for i in range(mask.shape[ax]):
            if ax == 0: filled[i] = binary_fill_holes(filled[i])
            elif ax == 1: filled[:, i, :] = binary_fill_holes(filled[:, i, :])
            else: filled[:, :, i] = binary_fill_holes(filled[:, :, i])
    return filled

def slicewise_morphology(mask, operation='close', iterations=1):
    result = mask.copy()
    struct_2d = generate_binary_structure(2, 1)
    for ax in range(3):
        for i in range(mask.shape[ax]):
            if ax == 0: slc = result[i]
            elif ax == 1: slc = result[:, i, :]
            else: slc = result[:, :, i]
            if operation == 'close':
                slc_new = binary_closing(slc, structure=struct_2d, iterations=iterations)
            elif operation == 'open':
                slc_new = binary_opening(slc, structure=struct_2d, iterations=iterations)
            else:
                slc_new = slc
            if ax == 0: result[i] = slc_new
            elif ax == 1: result[:, i, :] = slc_new
            else: result[:, :, i] = slc_new
    return result

def postprocess(prob, threshold=0.5, min_size=50,
                do_3d_close=True, do_3d_fill=True,
                do_slice_close=True, do_slice_fill=True, do_slice_open=True):
    """Parameterized post-processing. All ops are topology-safe."""
    mask = (prob > threshold).astype(np.uint8)
    if mask.sum() == 0:
        return mask

    mask = remove_small_components(mask, min_size)

    struct_3d = generate_binary_structure(3, 3)

    if do_3d_close:
        mask = topology_safe_op(mask,
            lambda m: binary_closing(m, structure=struct_3d, iterations=1).astype(np.uint8),
            "3D_close")

    if do_3d_fill:
        mask = topology_safe_op(mask,
            lambda m: binary_fill_holes(m).astype(np.uint8), "3D_fill")

    if do_slice_close:
        mask = topology_safe_op(mask,
            lambda m: slicewise_morphology(m, 'close', 1), "slice_close")

    if do_slice_fill:
        mask = topology_safe_op(mask, slicewise_hole_fill, "slice_fill")

    if do_3d_fill:
        mask = topology_safe_op(mask,
            lambda m: binary_fill_holes(m).astype(np.uint8), "3D_fill2")

    if do_slice_open:
        mask = topology_safe_op(mask,
            lambda m: slicewise_morphology(m, 'open', 1), "slice_open")

    mask = remove_small_components(mask, min_size)
    return mask


print("Competition metric + post-processing functions ready")
print(f"  Full metrics (Betti matching): {HAVE_FULL_METRICS}")
print(f"  LB = 0.30 × TopoScore + 0.35 × SurfaceDice + 0.35 × VOI_score")

In [ ]:
# =============================================================================
# CELL 7: GENERATE OOF PREDICTIONS — MEMORY & DISK EFFICIENT + DUAL T4
#
# CONSTRAINTS:
#   RAM  ~30 GB → process ONE volume at a time, never bulk-load
#   Disk  15 GB → save SWA .npy ONLY for eval subset; individual model
#                 predictions scored inline then discarded (never saved)
#   GPUs  2×T4  → sequential round-robin (no threading — avoids CUDA errors)
#
# FLOW (per fold):
#   1. Select eval_subset (MAX_EVAL_SAMPLES volumes, deterministic seed)
#   2. For each model (4 per fold):
#        Load model → move to GPU → convert to half precision ON GPU
#        Infer eval volumes sequentially, alternating GPUs
#        Score each immediately → append to model_comparison results
#        Accumulate into SWA dict (RAM, ~5 GB peak for 40 vols)
#        Unload model
#   3. Finalize SWA: divide accumulators, save float16 .npy, score SWA
#
# DISK USAGE: MAX_EVAL_SAMPLES × 2 folds × 65 MB ≈ 5 GB
# RAM PEAK:  SWA accumulators (MAX_EVAL_SAMPLES × 131 MB) + 1 vol in flight
# =============================================================================

def _test_cuda_devices(n_gpus):
    """Test each GPU with a small tensor op. Returns number of working GPUs."""
    working = 0
    for i in range(n_gpus):
        try:
            torch.cuda.set_device(i)
            # Small alloc + compute to verify device is functional
            x = torch.randn(64, 64, device=f'cuda:{i}')
            y = x @ x.T
            del x, y
            torch.cuda.synchronize(i)
            torch.cuda.empty_cache()
            print(f"  cuda:{i} — OK ({torch.cuda.get_device_name(i)})")
            working += 1
        except Exception as e:
            print(f"  cuda:{i} — FAILED: {e}")
            break  # Don't try higher GPUs if a lower one failed
    return working


def _load_model_to_gpu(ckpt_path, features, n_blocks, device):
    """
    Load checkpoint → CPU float32 → .to(device) → .half() ON GPU.
    Moving float32 first, then converting to half on GPU avoids
    half-precision CPU parameter issues that cause CUDA illegal access.
    """
    model, epoch, dice = load_model_from_checkpoint(
        ckpt_path, features, n_blocks, device='cpu'
    )
    # Move to GPU as float32 first
    model = model.to(device)
    torch.cuda.synchronize()
    # Convert to half precision ON GPU (safer than CPU half → GPU copy)
    model = model.half()
    torch.cuda.synchronize()
    return model, epoch, dice


def _infer_and_score_one(model, vid, device, images_dir, labels_dir,
                          patch_size, overlap, threshold=0.5, min_size=50):
    """
    Single volume: load → normalize → infer → score → return.
    Returns (vid, prob_float32, scores_dict, shape_str).
    """
    gpu_id = int(device.split(':')[1]) if ':' in str(device) else 0
    torch.cuda.set_device(gpu_id)

    img_path = images_dir / f"{vid}.tif"
    gt_path = labels_dir / f"{vid}.tif"

    # Load + normalize on CPU
    raw = tifffile.imread(str(img_path)).astype(np.float32)
    shape_str = str(raw.shape)
    normalized = robust_zscore_normalize(raw)
    del raw

    # Inference on GPU
    prob = sliding_window_inference(
        model, normalized, patch_size, overlap, device, verbose=True
    )
    del normalized
    torch.cuda.synchronize(gpu_id)
    torch.cuda.empty_cache()

    # Score on CPU
    gt = tifffile.imread(str(gt_path)).astype(np.uint8)
    mask = postprocess(prob, threshold=threshold, min_size=min_size)
    scores = compute_competition_score(mask, gt, ignore_label=2)
    scores['n_components'] = int(count_components(mask))
    scores['fg_pct'] = round(100.0 * mask.mean(), 2)
    del mask, gt
    gc.collect()

    return vid, prob.astype(np.float32), scores, shape_str


def generate_oof_predictions():
    """Memory-efficient OOF prediction with dual-GPU support + inline scoring."""
    global N_GPUS  # may reduce if a GPU fails the test

    if not FOLD_SPLITS:
        print("ERROR: No fold splits. Run Cell 3 first.")
        return

    images_dir = cfg.DATA_ROOT / "train_images"
    labels_dir = cfg.DATA_ROOT / "train_labels"

    # --- CUDA diagnostic: verify GPUs are functional ---
    print("CUDA device test:")
    n_working = _test_cuda_devices(N_GPUS)
    if n_working == 0:
        print("  No working GPUs! Falling back to CPU (VERY slow)")
        N_GPUS = 0
    elif n_working < N_GPUS:
        print(f"  Only {n_working}/{N_GPUS} GPUs working, using {n_working}")
        N_GPUS = n_working

    print(f"\nGPUs: {N_GPUS}")
    print(f"Eval subset per fold: {cfg.MAX_EVAL_SAMPLES} volumes")
    print(f"Disk budget: ~{cfg.MAX_EVAL_SAMPLES * 2 * 65 / 1000:.1f} GB for SWA .npy files")

    all_model_scores = []  # Collected for Cell 10

    for fold_idx in [0, 1]:
        fold_info = FOLD_MODELS[fold_idx]
        _, val_ids = FOLD_SPLITS[fold_idx]
        features = fold_info['features']
        n_blocks = fold_info['n_blocks']

        # --- Select eval subset (deterministic) ---
        valid_ids = [v for v in val_ids
                     if (images_dir / f"{v}.tif").exists() and
                        (labels_dir / f"{v}.tif").exists()]

        rng = np.random.RandomState(cfg.CV_SEED + fold_idx)
        if cfg.MAX_EVAL_SAMPLES > 0 and len(valid_ids) > cfg.MAX_EVAL_SAMPLES:
            eval_ids = sorted(
                rng.choice(valid_ids, cfg.MAX_EVAL_SAMPLES, replace=False).tolist()
            )
        else:
            eval_ids = sorted(valid_ids)

        print(f"\n{'='*70}")
        print(f"FOLD {fold_idx}: {len(valid_ids)} valid, evaluating {len(eval_ids)}")
        print(f"  Architecture: features={features}, blocks={n_blocks}")
        print(f"{'='*70}")

        # SWA accumulators kept in RAM (only for eval subset)
        swa_accum = {}   # vid → np.float32 array
        swa_count = {}   # vid → int

        for model_idx, (ckpt_path, model_name) in enumerate(
            zip(fold_info['paths'], fold_info['names'])
        ):
            print(f"\n  --- Model {model_idx+1}/4: {model_name} ---")

            if not Path(ckpt_path).exists():
                print(f"  SKIP: {ckpt_path} not found")
                continue

            # --- Load model(s) to GPU(s) ---
            # Strategy: float32 on CPU → .to(device) → .half() ON GPU
            # This avoids the "CUDA illegal memory access" from half-precision
            # CPU parameters being copied to GPU.
            models = {}
            try:
                torch.cuda.empty_cache()
                model0, epoch, dice = _load_model_to_gpu(
                    ckpt_path, features, n_blocks, 'cuda:0'
                )
                models[0] = model0
                print(f"    Epoch: {epoch}, Best Dice: {dice:.4f}")
                print(f"    Loaded to cuda:0")
            except Exception as e:
                print(f"    FATAL: Cannot load model to cuda:0: {e}")
                traceback.print_exc()
                continue

            if N_GPUS >= 2:
                try:
                    model1, _, _ = _load_model_to_gpu(
                        ckpt_path, features, n_blocks, 'cuda:1'
                    )
                    models[1] = model1
                    print(f"    Loaded to cuda:1")
                except Exception as e:
                    print(f"    WARNING: cuda:1 failed ({e}), using single GPU")

            n_active_gpus = len(models)
            torch.cuda.empty_cache()

            counter = 0
            t_start = time.time()

            # --- Process eval volumes SEQUENTIALLY, round-robin across GPUs ---
            for vol_i, vid in enumerate(eval_ids):
                gpu_id = vol_i % n_active_gpus
                device = f'cuda:{gpu_id}'

                try:
                    vid_out, prob, scores, shape_str = _infer_and_score_one(
                        models[gpu_id], vid, device,
                        images_dir, labels_dir,
                        cfg.PATCH_SIZE, cfg.OVERLAP
                    )
                except Exception as e:
                    print(f"    ERROR on {vid} (cuda:{gpu_id}): {e}")
                    traceback.print_exc()
                    # Try other GPU if available
                    if n_active_gpus > 1:
                        alt_gpu = 1 - gpu_id
                        try:
                            print(f"    Retrying on cuda:{alt_gpu}...")
                            vid_out, prob, scores, shape_str = _infer_and_score_one(
                                models[alt_gpu], vid, f'cuda:{alt_gpu}',
                                images_dir, labels_dir,
                                cfg.PATCH_SIZE, cfg.OVERLAP
                            )
                        except Exception as e2:
                            print(f"    SKIP {vid}: both GPUs failed: {e2}")
                            continue
                    else:
                        print(f"    SKIP {vid}")
                        continue

                # Accumulate SWA in main thread
                if vid in swa_accum:
                    swa_accum[vid] += prob
                else:
                    swa_accum[vid] = prob.copy()
                swa_count[vid] = swa_count.get(vid, 0) + 1
                del prob

                # Record individual model scores
                all_model_scores.append({
                    'fold': fold_idx, 'vol_id': vid,
                    'model': model_name, 'shape': shape_str,
                    **scores,
                })

                counter += 1
                elapsed = time.time() - t_start
                rate = counter / elapsed * 60 if elapsed > 0 else 0
                eta_min = (len(eval_ids) - counter) / (rate + 1e-9)
                print(f"    [{counter}/{len(eval_ids)}] {vid}: "
                      f"LB={scores['lb_score']:.4f} "
                      f"SDice={scores['surface_dice']:.4f} "
                      f"VOI={scores['voi_score']:.4f} "
                      f"({rate:.1f} vol/min, ETA {eta_min:.0f}m)")
                gc.collect()

            # Cleanup models from all GPUs
            for gpu_id in list(models.keys()):
                del models[gpu_id]
            del models
            for i in range(N_GPUS):
                torch.cuda.synchronize(i)
            torch.cuda.empty_cache()
            gc.collect()

            dt = time.time() - t_start
            print(f"  {model_name} done: {counter} vols in {dt/60:.1f} min")

        # --- Finalize SWA: average, save to disk as float16, score ---
        swa_dir = cfg.PRED_DIR / f"fold{fold_idx}_swa"
        swa_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n  SWA: finalizing {len(swa_accum)} volumes...")
        for vid in sorted(swa_accum.keys()):
            cnt = swa_count.get(vid, 1)
            swa_prob = swa_accum[vid] / cnt

            # Save float16 to disk (only ~65 MB per volume)
            swa_path = swa_dir / f"{vid}.npy"
            np.save(str(swa_path), swa_prob.astype(np.float16))

            # Score SWA at default threshold
            gt = tifffile.imread(str(labels_dir / f"{vid}.tif")).astype(np.uint8)
            mask = postprocess(swa_prob, threshold=0.5, min_size=50)
            scores = compute_competition_score(mask, gt, ignore_label=2)
            scores['n_components'] = int(count_components(mask))
            scores['fg_pct'] = round(100.0 * mask.mean(), 2)
            del mask, gt

            all_model_scores.append({
                'fold': fold_idx, 'vol_id': vid,
                'model': f'fold{fold_idx}_swa', 'shape': str(swa_prob.shape),
                **scores,
            })

            sz_mb = swa_path.stat().st_size / 1e6
            print(f"    {vid}: SWA LB={scores['lb_score']:.4f} "
                  f"(avg of {cnt} models, {sz_mb:.0f}MB)")

            del swa_prob

        # Free SWA accumulators
        del swa_accum, swa_count
        gc.collect()

        # Per-fold summary
        fold_df = pd.DataFrame([s for s in all_model_scores if s['fold'] == fold_idx])
        if len(fold_df) > 0:
            model_mean = fold_df.groupby('model')['lb_score'].mean().sort_values(ascending=False)
            print(f"\n  FOLD {fold_idx} MODEL RANKING:")
            for model, lb in model_mean.items():
                marker = " ★" if 'swa' in str(model) else ""
                print(f"    {str(model):<20s}: mean LB={lb:.4f}{marker}")

    # --- Save all model scores to CSV (for Cell 10) ---
    df_all = pd.DataFrame(all_model_scores)
    csv_path = cfg.OUTPUT_DIR / "model_comparison.csv"
    df_all.to_csv(csv_path, index=False)

    # Disk usage report
    total_disk_mb = sum(f.stat().st_size for f in cfg.PRED_DIR.rglob("*.npy")) / 1e6
    print(f"\n{'='*70}")
    print(f"OOF PREDICTION COMPLETE")
    print(f"  Model scores: {csv_path} ({len(df_all)} rows)")
    print(f"  SWA .npy files: {total_disk_mb:.0f} MB on disk")
    print(f"{'='*70}")

# Run it
generate_oof_predictions()


In [ ]:
# =============================================================================
# CELL 8: THRESHOLD SWEEP (on saved SWA .npy — one volume at a time)
#
# Loads SWA probability maps from disk one at a time.
# Tests thresholds × min_sizes using exact competition metric.
# =============================================================================

def sweep_thresholds():
    """Sweep thresholds on saved SWA OOF predictions."""
    labels_dir = cfg.DATA_ROOT / "train_labels"

    thresholds = np.arange(0.20, 0.75, 0.05)
    min_sizes = [30, 50, 100]

    all_results = []

    for fold_idx in [0, 1]:
        if fold_idx >= len(FOLD_SPLITS):
            continue

        swa_dir = cfg.PRED_DIR / f"fold{fold_idx}_swa"
        if not swa_dir.exists():
            print(f"Fold {fold_idx}: SWA dir not found, skipping")
            continue

        # Get available eval volumes (only those with saved SWA .npy)
        eval_ids = sorted([
            int(f.stem) for f in swa_dir.glob("*.npy")
            if (labels_dir / f"{f.stem}.tif").exists()
        ])

        print(f"\n{'='*70}")
        print(f"FOLD {fold_idx} — THRESHOLD SWEEP: {len(eval_ids)} volumes")
        print(f"  Thresholds: {[f'{t:.2f}' for t in thresholds]}")
        print(f"  Min sizes: {min_sizes}")
        print(f"{'='*70}")

        for vi, vid in enumerate(eval_ids):
            # Load ONE volume at a time
            prob = np.load(str(swa_dir / f"{vid}.npy")).astype(np.float32)
            gt = tifffile.imread(str(labels_dir / f"{vid}.tif")).astype(np.uint8)
            print(f"\n  [{vi+1}/{len(eval_ids)}] vol={vid}, shape={prob.shape}")

            for thresh in thresholds:
                for min_sz in min_sizes:
                    mask = postprocess(prob, threshold=thresh, min_size=min_sz,
                                       do_3d_close=True, do_3d_fill=True,
                                       do_slice_close=True, do_slice_fill=True,
                                       do_slice_open=True)

                    n_comp = count_components(mask)
                    fg_pct = 100 * mask.mean()
                    scores = compute_competition_score(mask, gt, ignore_label=2)
                    del mask

                    result = {
                        'fold': fold_idx, 'vol_id': vid,
                        'threshold': round(thresh, 2), 'min_size': min_sz,
                        'n_components': n_comp, 'fg_pct': round(fg_pct, 2),
                        **scores,
                    }
                    all_results.append(result)

                    if min_sz == 50:
                        print(f"    T={thresh:.2f} ms={min_sz:3d} | "
                              f"LB={scores['lb_score']:.4f} "
                              f"SDice={scores['surface_dice']:.4f} "
                              f"VOI={scores['voi_score']:.4f} | "
                              f"comp={n_comp} FG={fg_pct:.1f}%")

            del prob, gt
            gc.collect()

    df = pd.DataFrame(all_results)

    if len(df) > 0:
        print(f"\n{'='*70}")
        print("THRESHOLD SWEEP RESULTS")
        print(f"{'='*70}")

        for ms in min_sizes:
            sub = df[df['min_size'] == ms]
            if len(sub) == 0: continue
            mean_scores = sub.groupby('threshold')['lb_score'].mean()
            best_t = mean_scores.idxmax()
            best_s = mean_scores.max()
            print(f"\n  min_size={ms}:")
            for t in sorted(mean_scores.index):
                marker = " <<<" if t == best_t else ""
                print(f"    threshold={t:.2f}: mean_LB={mean_scores[t]:.4f}{marker}")

        # Overall best
        combo_mean = df.groupby(['threshold', 'min_size'])['lb_score'].mean()
        best_combo = combo_mean.idxmax()
        best_lb = combo_mean.max()
        print(f"\n  *** BEST: threshold={best_combo[0]}, min_size={best_combo[1]}, "
              f"mean_LB={best_lb:.4f} ***")

    csv_path = cfg.OUTPUT_DIR / "threshold_sweep.csv"
    df.to_csv(csv_path, index=False)
    print(f"\nSaved: {csv_path}")

    return df

threshold_results = sweep_thresholds()

In [ ]:
# =============================================================================
# CELL 9: POST-PROCESSING ABLATION (one volume at a time from disk)
#
# Tests all 2^5 = 32 combinations of post-processing flags
# at the best threshold from Cell 8.
# =============================================================================

def ablate_postprocessing(best_threshold=0.50, best_min_size=50):
    """Test all 32 post-processing combinations on SWA predictions."""
    labels_dir = cfg.DATA_ROOT / "train_labels"

    # Use best threshold from sweep if available
    if 'threshold_results' in dir() and threshold_results is not None and len(threshold_results) > 0:
        combo_mean = threshold_results.groupby(['threshold', 'min_size'])['lb_score'].mean()
        best_combo = combo_mean.idxmax()
        best_threshold = best_combo[0]
        best_min_size = best_combo[1]
        print(f"Using best from sweep: threshold={best_threshold}, min_size={best_min_size}")

    pp_flags = ['3d_close', '3d_fill', 'slice_close', 'slice_fill', 'slice_open']
    combinations = list(product([True, False], repeat=5))
    print(f"Testing {len(combinations)} post-processing combinations")

    all_results = []

    for fold_idx in [0, 1]:
        swa_dir = cfg.PRED_DIR / f"fold{fold_idx}_swa"
        if not swa_dir.exists():
            continue

        eval_ids = sorted([
            int(f.stem) for f in swa_dir.glob("*.npy")
            if (labels_dir / f"{f.stem}.tif").exists()
        ])

        print(f"\n  Fold {fold_idx}: {len(eval_ids)} volumes × {len(combinations)} combos")

        for vi, vid in enumerate(eval_ids):
            prob = np.load(str(swa_dir / f"{vid}.npy")).astype(np.float32)
            gt = tifffile.imread(str(labels_dir / f"{vid}.tif")).astype(np.uint8)

            for combo in combinations:
                mask = postprocess(prob, threshold=best_threshold, min_size=best_min_size,
                                   do_3d_close=combo[0], do_3d_fill=combo[1],
                                   do_slice_close=combo[2], do_slice_fill=combo[3],
                                   do_slice_open=combo[4])

                scores = compute_competition_score(mask, gt, ignore_label=2)
                result = {
                    'fold': fold_idx, 'vol_id': vid,
                    'threshold': best_threshold, 'min_size': best_min_size,
                    '3d_close': combo[0], '3d_fill': combo[1],
                    'slice_close': combo[2], 'slice_fill': combo[3],
                    'slice_open': combo[4],
                    **scores,
                }
                all_results.append(result)
                del mask

            del prob, gt
            gc.collect()
            print(f"    [{vi+1}/{len(eval_ids)}] {vid} done")

    df = pd.DataFrame(all_results)

    if len(df) > 0:
        df['combo'] = df.apply(
            lambda r: f"3dc={int(r['3d_close'])} 3df={int(r['3d_fill'])} "
                      f"sc={int(r['slice_close'])} sf={int(r['slice_fill'])} "
                      f"so={int(r['slice_open'])}",
            axis=1
        )

        combo_scores = df.groupby('combo').agg({
            'lb_score': 'mean', 'toposcore': 'mean',
            'surface_dice': 'mean', 'voi_score': 'mean'
        }).sort_values('lb_score', ascending=False)

        print(f"\n{'='*70}")
        print("POST-PROCESSING ABLATION (sorted by mean LB)")
        print(f"{'='*70}")
        print(f"{'Combo':<45} {'LB':>8} {'Topo':>8} {'SDice':>8} {'VOI':>8}")
        print("-" * 80)
        for combo, row in combo_scores.head(10).iterrows():
            print(f"{combo:<45} {row['lb_score']:8.4f} {row['toposcore']:8.4f} "
                  f"{row['surface_dice']:8.4f} {row['voi_score']:8.4f}")

        # Individual step impact
        print(f"\n  INDIVIDUAL STEP IMPACT:")
        baseline = df[(~df['3d_close']) & (~df['3d_fill']) & (~df['slice_close']) &
                       (~df['slice_fill']) & (~df['slice_open'])]['lb_score'].mean()
        print(f"  Baseline (no post-proc): {baseline:.4f}")
        for flag in pp_flags:
            with_flag = df[df[flag] == True]['lb_score'].mean()
            without_flag = df[df[flag] == False]['lb_score'].mean()
            delta = with_flag - without_flag
            arrow = "↑" if delta > 0.001 else ("↓" if delta < -0.001 else "≈")
            print(f"    {flag:15s}: Δ={delta:+.4f} {arrow}")

    csv_path = cfg.OUTPUT_DIR / "postprocess_ablation.csv"
    df.to_csv(csv_path, index=False)
    print(f"\nSaved: {csv_path}")
    return df

pp_results = ablate_postprocessing()

In [ ]:
# =============================================================================
# CELL 10: OVERLAP SWEEP (re-infer small subset at different overlap values)
#
# Overlap affects how patches overlap during sliding window inference.
# Higher overlap = more patch averaging = smoother, potentially better
# predictions, but SLOWER inference (more patches per volume).
#
# Patches per 320³ volume at each overlap:
#   0.25 → ~8 patches    (fastest, least averaging)
#   0.375 → ~15 patches
#   0.50 → ~27 patches   (current default)
#   0.625 → ~64 patches
#   0.75 → ~125 patches  (slowest, most averaging)
#
# We test on a SMALL subset (5 volumes) with fold0_best to find the sweet
# spot between quality and inference speed.
# =============================================================================

def overlap_sweep():
    """Test different overlap values on a small volume subset."""
    
    images_dir = cfg.DATA_ROOT / "train_images"
    labels_dir = cfg.DATA_ROOT / "train_labels"
    
    # Use fold 0 eval volumes (same deterministic subset as Cell 7)
    _, val_ids = FOLD_SPLITS[0]
    valid_ids = [v for v in val_ids
                 if (images_dir / f"{v}.tif").exists() and
                    (labels_dir / f"{v}.tif").exists()]
    
    rng = np.random.RandomState(cfg.CV_SEED)
    if len(valid_ids) > cfg.MAX_EVAL_SAMPLES:
        eval_ids = sorted(
            rng.choice(valid_ids, cfg.MAX_EVAL_SAMPLES, replace=False).tolist()
        )
    else:
        eval_ids = sorted(valid_ids)
    
    # Use only first 5 volumes for speed
    N_OVERLAP_VOLS = min(5, len(eval_ids))
    sweep_ids = eval_ids[:N_OVERLAP_VOLS]
    
    OVERLAPS_TO_TEST = [0.25, 0.375, 0.5, 0.625]
    
    print(f"OVERLAP SWEEP")
    print(f"  Volumes: {N_OVERLAP_VOLS} from fold 0")
    print(f"  Overlaps: {OVERLAPS_TO_TEST}")
    print(f"  Model: fold0_best")
    print(f"  Estimated time: ~{N_OVERLAP_VOLS * len(OVERLAPS_TO_TEST) * 2.5:.0f} min")
    
    # Load model to GPU 0
    ckpt_path = cfg.FOLD0_BEST
    if not Path(ckpt_path).exists():
        print(f"ERROR: {ckpt_path} not found")
        return
    
    features = cfg.FOLD0_FEATURES
    n_blocks = cfg.FOLD0_N_BLOCKS
    
    torch.cuda.empty_cache()
    model, epoch, dice = load_model_from_checkpoint(
        ckpt_path, features, n_blocks, device='cpu'
    )
    model = model.to('cuda:0').half()
    torch.cuda.synchronize()
    print(f"  Model loaded: epoch={epoch}, dice={dice:.4f}")
    
    results = []
    
    for ov in OVERLAPS_TO_TEST:
        print(f"\n  --- Overlap = {ov} ---")
        t0 = time.time()
        
        for vi, vid in enumerate(sweep_ids):
            img_path = images_dir / f"{vid}.tif"
            gt_path = labels_dir / f"{vid}.tif"
            
            # Load + normalize
            raw = tifffile.imread(str(img_path)).astype(np.float32)
            normalized = robust_zscore_normalize(raw)
            shape_str = str(raw.shape)
            del raw
            
            # Inference with this overlap
            prob = sliding_window_inference(
                model, normalized, cfg.PATCH_SIZE, ov, 'cuda:0', verbose=True
            )
            del normalized
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            
            # Score
            gt = tifffile.imread(str(gt_path)).astype(np.uint8)
            mask = postprocess(prob, threshold=0.5, min_size=50)
            scores = compute_competition_score(mask, gt, ignore_label=2)
            del prob, mask, gt
            gc.collect()
            
            results.append({
                'overlap': ov,
                'vol_id': vid,
                'shape': shape_str,
                'lb_score': scores['lb_score'],
                'toposcore': scores['toposcore'],
                'surface_dice': scores['surface_dice'],
                'voi_score': scores['voi_score'],
            })
            
            print(f"    [{vi+1}/{N_OVERLAP_VOLS}] {vid}: LB={scores['lb_score']:.4f} "
                  f"SDice={scores['surface_dice']:.4f} VOI={scores['voi_score']:.4f}")
        
        dt = time.time() - t0
        print(f"    Time: {dt/60:.1f} min")
    
    # Cleanup model
    del model
    torch.cuda.empty_cache()
    gc.collect()
    
    # Results summary
    df = pd.DataFrame(results)
    csv_path = cfg.OUTPUT_DIR / "overlap_sweep.csv"
    df.to_csv(csv_path, index=False)
    
    print(f"\n{'='*70}")
    print(f"OVERLAP SWEEP RESULTS ({N_OVERLAP_VOLS} volumes, fold0_best)")
    print(f"{'='*70}")
    print(f"\n  {'Overlap':>8} {'Mean LB':>10} {'Topo':>8} {'SDice':>8} {'VOI':>8} {'Patches':>9}")
    print(f"  {'-'*60}")
    
    overlap_summary = df.groupby('overlap').agg({
        'lb_score': 'mean', 'toposcore': 'mean',
        'surface_dice': 'mean', 'voi_score': 'mean',
    }).sort_values('lb_score', ascending=False)
    
    best_ov = overlap_summary.index[0]
    for ov, row in overlap_summary.iterrows():
        # Estimate patches for a 320³ volume
        stride = int(192 * (1 - ov))
        n_per_dim = max(1, (320 - 192) // stride + 1) if stride > 0 else 1
        est_patches = n_per_dim ** 3
        marker = " ★" if ov == best_ov else ""
        print(f"  {ov:8.3f} {row['lb_score']:10.4f} {row['toposcore']:8.4f} "
              f"{row['surface_dice']:8.4f} {row['voi_score']:8.4f} {est_patches:>7d}{marker}")
    
    best_lb = overlap_summary.loc[best_ov, 'lb_score']
    default_lb = overlap_summary.loc[0.5, 'lb_score'] if 0.5 in overlap_summary.index else best_lb
    delta = best_lb - default_lb
    
    print(f"\n  ★ Best overlap: {best_ov}")
    print(f"    Mean LB = {best_lb:.4f}")
    if best_ov != 0.5:
        print(f"    vs default 0.5: Δ = {delta:+.4f}")
    else:
        print(f"    Default 0.5 is already optimal!")
    
    print(f"\n  Saved: {csv_path}")
    return df

overlap_results = overlap_sweep()


In [ ]:
# =============================================================================
# CELL 11: MODEL COMPARISON (loads CSV from Cell 7 — no re-inference)
#
# Cell 7 already scored every model + SWA inline during OOF prediction.
# This cell just loads the results and presents them clearly.
# =============================================================================

def display_model_comparison():
    """Display model comparison from pre-computed CSV."""
    csv_path = cfg.OUTPUT_DIR / "model_comparison.csv"

    if not csv_path.exists():
        print("ERROR: model_comparison.csv not found. Run Cell 7 first.")
        return pd.DataFrame()

    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} score entries from {csv_path}")

    if len(df) == 0:
        return df

    # --- Overall model ranking ---
    print(f"\n{'='*70}")
    print("MODEL COMPARISON (sorted by mean LB)")
    print(f"{'='*70}")

    model_scores = df.groupby('model').agg({
        'lb_score': ['mean', 'std', 'count'],
        'toposcore': 'mean',
        'surface_dice': 'mean',
        'voi_score': 'mean'
    })
    model_scores.columns = ['lb_mean', 'lb_std', 'n_vols', 'topo', 'sdice', 'voi']
    model_scores = model_scores.sort_values('lb_mean', ascending=False)

    print(f"\n{'Model':<25} {'LB':>8} {'±std':>7} {'N':>4} {'Topo':>8} {'SDice':>8} {'VOI':>8}")
    print("-" * 75)
    for model, row in model_scores.iterrows():
        marker = " ★" if 'swa' in str(model) else ""
        print(f"{str(model):<25} {row['lb_mean']:8.4f} {row['lb_std']:7.4f} "
              f"{int(row['n_vols']):4d} {row['topo']:8.4f} "
              f"{row['sdice']:8.4f} {row['voi']:8.4f}{marker}")

    # --- SWA vs best individual per fold ---
    print(f"\n\nSWA vs BEST INDIVIDUAL:")
    for fold_idx in [0, 1]:
        fold_df = df[df['fold'] == fold_idx]
        if len(fold_df) == 0:
            continue

        swa_df = fold_df[fold_df['model'].str.contains('swa')]
        indiv_df = fold_df[~fold_df['model'].str.contains('swa')]

        if len(swa_df) > 0 and len(indiv_df) > 0:
            swa_lb = swa_df['lb_score'].mean()
            indiv_mean = indiv_df.groupby('model')['lb_score'].mean()
            best_indiv_lb = indiv_mean.max()
            best_indiv_name = indiv_mean.idxmax()
            delta = swa_lb - best_indiv_lb
            verdict = "SWA WINS ✓" if delta > 0 else "Individual WINS"
            print(f"  Fold {fold_idx}: SWA={swa_lb:.4f} vs {best_indiv_name}={best_indiv_lb:.4f} "
                  f"→ Δ={delta:+.4f} ({verdict})")

    # --- Per-volume detail for best and worst ---
    if 'vol_id' in df.columns:
        swa_vols = df[df['model'].str.contains('swa')].copy()
        if len(swa_vols) > 0:
            swa_vols_sorted = swa_vols.sort_values('lb_score')
            print(f"\n  SWA: worst 3 volumes:")
            for _, row in swa_vols_sorted.head(3).iterrows():
                print(f"    vol={row['vol_id']}: LB={row['lb_score']:.4f} "
                      f"Topo={row['toposcore']:.4f} SDice={row['surface_dice']:.4f}")
            print(f"  SWA: best 3 volumes:")
            for _, row in swa_vols_sorted.tail(3).iterrows():
                print(f"    vol={row['vol_id']}: LB={row['lb_score']:.4f} "
                      f"Topo={row['toposcore']:.4f} SDice={row['surface_dice']:.4f}")

    csv_save = cfg.OUTPUT_DIR / "model_comparison.csv"
    print(f"\nResults at: {csv_save}")
    return df

model_results = display_model_comparison()

In [ ]:
# =============================================================================
# CELL 12: RESULTS SUMMARY + BEST CONFIG EXPORT
#
# Aggregates findings from:
#   - Cell 8: Threshold sweep → optimal threshold + min_size
#   - Cell 9: Post-processing ablation → best combination of 5 flags
#   - Cell 11: Model comparison → SWA vs individual, per-model ranking
#
# Exports: optimal_config.json for use in submission notebook
# =============================================================================

def build_results_summary():
    """Aggregate all CV evaluation results into a single report + export config."""

    print("=" * 70)
    print("  VESUVIUS 2025 — CV EVALUATION SUMMARY")
    print("=" * 70)

    best_config = {
        'threshold': 0.50,
        'min_component_size': 50,
        'overlap': 0.50,
        'postprocess': {
            '3d_close': True,
            '3d_fill': True,
            'slice_close': True,
            'slice_fill': True,
            'slice_open': True,
        },
        'use_swa': True,
        'metrics': {},
        'per_fold': {},
    }

    # =========================================================================
    # 1. THRESHOLD SWEEP RESULTS
    # =========================================================================
    thresh_csv = cfg.OUTPUT_DIR / "threshold_sweep.csv"
    if thresh_csv.exists():
        df_t = pd.read_csv(thresh_csv)
        print(f"\n{'─'*70}")
        print("1. THRESHOLD SWEEP")
        print(f"{'─'*70}")

        # Best threshold × min_size combination
        combo_mean = df_t.groupby(['threshold', 'min_size']).agg({
            'lb_score': 'mean',
            'toposcore': 'mean',
            'surface_dice': 'mean',
            'voi_score': 'mean',
        }).reset_index()

        best_idx = combo_mean['lb_score'].idxmax()
        best_row = combo_mean.loc[best_idx]
        best_config['threshold'] = float(best_row['threshold'])
        best_config['min_component_size'] = int(best_row['min_size'])

        print(f"\n  ★ Best: threshold={best_row['threshold']:.2f}, "
              f"min_size={int(best_row['min_size'])}")
        print(f"    Mean LB = {best_row['lb_score']:.4f}")
        print(f"    Topo={best_row['toposcore']:.4f}  SDice={best_row['surface_dice']:.4f}  "
              f"VOI={best_row['voi_score']:.4f}")

        # Show top 5 combos
        top5 = combo_mean.sort_values('lb_score', ascending=False).head(5)
        print(f"\n  Top 5 threshold × min_size combinations:")
        print(f"  {'Thresh':>7} {'MinSz':>6} {'LB':>8} {'Topo':>8} {'SDice':>8} {'VOI':>8}")
        for _, r in top5.iterrows():
            marker = " ★" if (r['threshold'] == best_row['threshold'] and
                              r['min_size'] == best_row['min_size']) else ""
            print(f"  {r['threshold']:7.2f} {int(r['min_size']):6d} "
                  f"{r['lb_score']:8.4f} {r['toposcore']:8.4f} "
                  f"{r['surface_dice']:8.4f} {r['voi_score']:8.4f}{marker}")

        # Per-fold breakdown at best threshold
        best_t = best_row['threshold']
        best_ms = best_row['min_size']
        print(f"\n  Per-fold at best threshold ({best_t:.2f}, min_size={int(best_ms)}):")
        for fold in sorted(df_t['fold'].unique()):
            fold_sub = df_t[(df_t['fold'] == fold) &
                            (df_t['threshold'] == best_t) &
                            (df_t['min_size'] == best_ms)]
            if len(fold_sub) > 0:
                m = fold_sub.mean(numeric_only=True)
                best_config['per_fold'][f'fold_{fold}'] = {
                    'lb_score': round(float(m['lb_score']), 4),
                    'toposcore': round(float(m['toposcore']), 4),
                    'surface_dice': round(float(m['surface_dice']), 4),
                    'voi_score': round(float(m['voi_score']), 4),
                }
                print(f"    Fold {fold}: LB={m['lb_score']:.4f} "
                      f"(Topo={m['toposcore']:.4f} SDice={m['surface_dice']:.4f} "
                      f"VOI={m['voi_score']:.4f})")
    else:
        print("\n  Threshold sweep not found. Run Cell 8 first.")

    # =========================================================================
    # 2. POST-PROCESSING ABLATION RESULTS
    # =========================================================================
    pp_csv = cfg.OUTPUT_DIR / "postprocess_ablation.csv"
    if pp_csv.exists():
        df_pp = pd.read_csv(pp_csv)
        print(f"\n{'─'*70}")
        print("2. POST-PROCESSING ABLATION")
        print(f"{'─'*70}")

        pp_flags = ['3d_close', '3d_fill', 'slice_close', 'slice_fill', 'slice_open']

        # Best combination
        combo_cols = pp_flags + ['lb_score', 'toposcore', 'surface_dice', 'voi_score']
        combo_mean = df_pp.groupby(pp_flags).agg({
            'lb_score': 'mean', 'toposcore': 'mean',
            'surface_dice': 'mean', 'voi_score': 'mean',
        }).reset_index().sort_values('lb_score', ascending=False)

        best_pp = combo_mean.iloc[0]
        for flag in pp_flags:
            best_config['postprocess'][flag] = bool(best_pp[flag])

        active = [f for f in pp_flags if best_pp[f]]
        inactive = [f for f in pp_flags if not best_pp[f]]
        print(f"\n  ★ Best combination (LB={best_pp['lb_score']:.4f}):")
        print(f"    ON:  {', '.join(active) if active else 'none'}")
        print(f"    OFF: {', '.join(inactive) if inactive else 'none'}")

        # Baseline (no post-processing)
        no_pp = df_pp[(~df_pp['3d_close']) & (~df_pp['3d_fill']) &
                       (~df_pp['slice_close']) & (~df_pp['slice_fill']) &
                       (~df_pp['slice_open'])]
        if len(no_pp) > 0:
            baseline_lb = no_pp['lb_score'].mean()
            delta = best_pp['lb_score'] - baseline_lb
            print(f"\n  Baseline (no post-proc): LB={baseline_lb:.4f}")
            print(f"  Best post-proc adds:    Δ={delta:+.4f}")

        # Individual step impact
        print(f"\n  Individual step impact (average across all combos):")
        for flag in pp_flags:
            on_mean = df_pp[df_pp[flag] == True]['lb_score'].mean()
            off_mean = df_pp[df_pp[flag] == False]['lb_score'].mean()
            delta = on_mean - off_mean
            arrow = "↑" if delta > 0.001 else ("↓" if delta < -0.001 else "≈")
            print(f"    {flag:15s}: Δ={delta:+.4f} {arrow}")
    else:
        print("\n  Post-processing ablation not found. Run Cell 9 first.")

    # =========================================================================
    # 3. MODEL COMPARISON RESULTS
    # =========================================================================
    model_csv = cfg.OUTPUT_DIR / "model_comparison.csv"
    if model_csv.exists():
        df_m = pd.read_csv(model_csv)
        print(f"\n{'─'*70}")
        print("3. MODEL COMPARISON")
        print(f"{'─'*70}")

        model_scores = df_m.groupby('model').agg({
            'lb_score': 'mean', 'toposcore': 'mean',
            'surface_dice': 'mean', 'voi_score': 'mean',
        }).sort_values('lb_score', ascending=False)

        print(f"\n  {'Model':<25} {'LB':>8} {'Topo':>8} {'SDice':>8} {'VOI':>8}")
        print(f"  {'-'*65}")
        for model, row in model_scores.iterrows():
            marker = " ★" if 'swa' in str(model) else ""
            print(f"  {str(model):<25} {row['lb_score']:8.4f} {row['toposcore']:8.4f} "
                  f"{row['surface_dice']:8.4f} {row['voi_score']:8.4f}{marker}")

        # SWA vs best individual per fold
        print(f"\n  SWA vs Best Individual:")
        for fold_idx in [0, 1]:
            fold_df = df_m[df_m['fold'] == fold_idx]
            if len(fold_df) == 0:
                continue
            swa_df = fold_df[fold_df['model'].str.contains('swa')]
            indiv_df = fold_df[~fold_df['model'].str.contains('swa')]
            if len(swa_df) > 0 and len(indiv_df) > 0:
                swa_lb = swa_df['lb_score'].mean()
                indiv_mean = indiv_df.groupby('model')['lb_score'].mean()
                best_indiv_lb = indiv_mean.max()
                best_indiv_name = indiv_mean.idxmax()
                delta = swa_lb - best_indiv_lb
                verdict = "SWA WINS" if delta > 0 else "Individual WINS"
                best_config['per_fold'][f'fold_{fold_idx}_swa_vs_best'] = {
                    'swa': round(float(swa_lb), 4),
                    'best_individual': round(float(best_indiv_lb), 4),
                    'best_individual_name': str(best_indiv_name),
                    'delta': round(float(delta), 4),
                }
                print(f"    Fold {fold_idx}: SWA={swa_lb:.4f} vs {best_indiv_name}={best_indiv_lb:.4f} "
                      f"→ Δ={delta:+.4f} ({verdict})")

        # Overall best model
        best_model = model_scores.index[0]
        best_lb = model_scores.iloc[0]['lb_score']
        best_config['use_swa'] = 'swa' in str(best_model)
        best_config['metrics']['mean_lb'] = round(float(best_lb), 4)
        best_config['metrics']['mean_toposcore'] = round(float(model_scores.iloc[0]['toposcore']), 4)
        best_config['metrics']['mean_surface_dice'] = round(float(model_scores.iloc[0]['surface_dice']), 4)
        best_config['metrics']['mean_voi_score'] = round(float(model_scores.iloc[0]['voi_score']), 4)
    else:
        print("\n  Model comparison not found. Run Cell 11 first.")

    # =========================================================================
    # 4. OVERLAP SWEEP RESULTS
    # =========================================================================
    ov_csv = cfg.OUTPUT_DIR / "overlap_sweep.csv"
    if ov_csv.exists():
        df_ov = pd.read_csv(ov_csv)
        print(f"\n{'─'*70}")
        print("4. OVERLAP SWEEP")
        print(f"{'─'*70}")

        ov_mean = df_ov.groupby('overlap')['lb_score'].mean().sort_values(ascending=False)
        best_ov = ov_mean.index[0]
        best_config['overlap'] = float(best_ov)

        print(f"\n  {'Overlap':>8} {'Mean LB':>10}")
        print(f"  {'-'*22}")
        for ov, lb in ov_mean.items():
            marker = " ★" if ov == best_ov else ""
            print(f"  {ov:8.3f} {lb:10.4f}{marker}")

        if best_ov != 0.5:
            delta = ov_mean.loc[best_ov] - ov_mean.get(0.5, ov_mean.loc[best_ov])
            print(f"\n  ★ Best overlap: {best_ov} (Δ={delta:+.4f} vs 0.5)")
        else:
            print(f"\n  ★ Default overlap 0.5 is optimal")
    else:
        print("\n  Overlap sweep not found. Run Cell 11 first.")

    # =========================================================================
    # 5. FINAL SUMMARY
    # =========================================================================
    print(f"\n{'='*70}")
    print("  OPTIMAL CONFIGURATION FOR SUBMISSION")
    print(f"{'='*70}")
    print(f"""
  Threshold:          {best_config['threshold']}
  Min component size: {best_config['min_component_size']}
  Overlap:            {best_config['overlap']}
  Use SWA:            {best_config['use_swa']}

  Post-processing:
    3D closing:       {best_config['postprocess']['3d_close']}
    3D hole fill:     {best_config['postprocess']['3d_fill']}
    Slice closing:    {best_config['postprocess']['slice_close']}
    Slice hole fill:  {best_config['postprocess']['slice_fill']}
    Slice opening:    {best_config['postprocess']['slice_open']}

  Expected LB Score:  {best_config['metrics'].get('mean_lb', 'N/A')}
    TopoScore:        {best_config['metrics'].get('mean_toposcore', 'N/A')}
    SurfaceDice:      {best_config['metrics'].get('mean_surface_dice', 'N/A')}
    VOI Score:        {best_config['metrics'].get('mean_voi_score', 'N/A')}
""")

    # =========================================================================
    # 5. EXPORT JSON (for import into submission notebook)
    # =========================================================================
    config_path = cfg.OUTPUT_DIR / "optimal_config.json"
    with open(config_path, 'w') as f:
        json.dump(best_config, f, indent=2, default=str)
    print(f"  Config exported: {config_path}")

    # Also print as Python dict for quick copy-paste
    print(f"\n  Copy-paste for submission notebook:")
    print(f"  ─────────────────────────────────────")
    print(f"  THRESHOLD = {best_config['threshold']}")
    print(f"  MIN_COMPONENT_SIZE = {best_config['min_component_size']}")
    print(f"  OVERLAP = {best_config['overlap']}")
    pp = best_config['postprocess']
    print(f"  DO_3D_CLOSE = {pp['3d_close']}")
    print(f"  DO_3D_FILL = {pp['3d_fill']}")
    print(f"  DO_SLICE_CLOSE = {pp['slice_close']}")
    print(f"  DO_SLICE_FILL = {pp['slice_fill']}")
    print(f"  DO_SLICE_OPEN = {pp['slice_open']}")

    print(f"\n{'='*70}")
    print("  CV EVALUATION COMPLETE")
    print(f"{'='*70}")

    return best_config

optimal_config = build_results_summary()

# Vesuvius 2025 — CV Evaluation with Competition Metrics

**Purpose**: Evaluate 8 models (4 per fold) on OOF validation data using the EXACT competition metric.

## Pipeline
1. Build Betti-Matching-3D + install topometrics
2. Reproduce exact 3-fold splits (same StratifiedKFold)
3. Load 8 models → generate OOF probability maps → save to disk
4. Sweep threshold, post-processing, ensemble weights on saved predictions
5. Report optimal config for test submission

## Models (8 total)
| # | Fold | Type | Architecture | Params |
|---|------|------|-------------|--------|
| 1 | 0 | best_model | [320,320] bottleneck, [6,6] blocks | ~10M |
| 2 | 0 | checkpoint A | same | ~10M |
| 3 | 0 | checkpoint B | same | ~10M |
| 4 | 0 | checkpoint C | same | ~10M |
| 5 | 1 | best_model | [512,512] bottleneck, [4,4] blocks | ~18M |
| 6 | 1 | checkpoint A | same | ~18M |
| 7 | 1 | checkpoint B | same | ~18M |
| 8 | 1 | checkpoint C | same | ~18M |

## Scoring
```
LB = 0.30 × TopoScore + 0.35 × SurfaceDice + 0.35 × VOI_score
```